In [ ]:
import os
import pandas as pd
import numpy as np

srfdata_dir = '/share/home/dq013/zhwei/colm/data/CoLM_Forcing/PLUMBER2/Srfdata/'
metdata_dir = '/share/home/dq013/zhwei/colm/data/CoLM_Forcing/PLUMBER2/Forcing/'

srfdata_name_all = os.listdir(srfdata_dir)
metdata_name_all = os.listdir(metdata_dir)

srfdata_name_filtered = [s for s in srfdata_name_all if 'FLUXNET2015' in s]
metdata_name_filtered = [s for s in metdata_name_all if 'FLUXNET2015' in s]

df = pd.DataFrame(np.zeros((14,3)), columns=['SiteName','Syear','Eyear','Source','srfdata_dir','srfdata_name','metdata_dir','metdata_name','OBS_PFT','IGBP_CLASS','longitude','latitude'])

print(len(srfdata_name_filtered))
print(len(metdata_name_filtered))
for i in range(len(srfdata_name_filtered)):
    print(srfdata_name_filtered[i])

102
102
SD-Dem_2005-2009_FLUXNET2015_Srf.nc
FI-Sod_2008-2014_FLUXNET2015_Srf.nc
FR-Fon_2005-2013_FLUXNET2015_Srf.nc
RU-Che_2003-2004_FLUXNET2015_Srf.nc
US-SRG_2009-2014_FLUXNET2015_Srf.nc
CH-Cha_2006-2014_FLUXNET2015_Srf.nc
US-Var_2001-2014_FLUXNET2015_Srf.nc
IT-Cpz_2001-2008_FLUXNET2015_Srf.nc
CH-Oe1_2002-2008_FLUXNET2015_Srf.nc
IT-Lav_2005-2014_FLUXNET2015_Srf.nc
ZA-Kru_2000-2002_FLUXNET2015_Srf.nc
CN-Din_2003-2005_FLUXNET2015_Srf.nc
US-Ha1_1992-2012_FLUXNET2015_Srf.nc
US-MMS_1999-2014_FLUXNET2015_Srf.nc
DE-Hai_2000-2012_FLUXNET2015_Srf.nc
US-Whs_2008-2014_FLUXNET2015_Srf.nc
US-Cop_2002-2003_FLUXNET2015_Srf.nc
DK-ZaH_2000-2013_FLUXNET2015_Srf.nc
DK-Sor_1997-2014_FLUXNET2015_Srf.nc
US-ARM_2003-2012_FLUXNET2015_Srf.nc
CA-SF1_2004-2006_FLUXNET2015_Srf.nc
US-WCr_1999-2006_FLUXNET2015_Srf.nc
US-SRM_2004-2014_FLUXNET2015_Srf.nc
FR-Gri_2005-2013_FLUXNET2015_Srf.nc
CN-HaM_2002-2003_FLUXNET2015_Srf.nc
US-Ne1_2002-2012_FLUXNET2015_Srf.nc
IT-Ro2_2002-2008_FLUXNET2015_Srf.nc
US-Ne3_2002-2012_FLU

In [9]:
import os
import pandas as pd
from netCDF4 import Dataset

# Define directories
forcing_dir = '/share/home/dq013/zhwei/colm/data/CoLM_Forcing/PLUMBER2/Forcing'
srfdata_dir = '/share/home/dq013/zhwei/colm/data/CoLM_Forcing/PLUMBER2/Srfdata'

# Columns for the statistics table
columns = ['SiteName', 'Syear', 'Eyear', 'Source', 'srfdata_dir', 'srfdata_name', 
           'metdata_dir', 'metdata_name', 'IGBP_CLASS', 'longitude', 'latitude']

# List to hold the rows
rows = []

# List all files in forcing_dir
for filename in os.listdir(srfdata_dir):
    if filename.endswith('_Srf.nc'):
        # Parse filename
        parts = filename.split('_')
        if len(parts) != 4 or parts[3] != 'Srf.nc':
            print(f"Skipping invalid file: {filename}")
            continue
        
        site_name = parts[0]
        years_str = parts[1]
        source = parts[2]
        
        if '-' not in years_str:
            print(f"Skipping invalid years in file: {filename}")
            continue
        
        syear_str, eyear_str = years_str.split('-')
        try:
            syear = int(syear_str)
            eyear = int(eyear_str)
        except ValueError:
            print(f"Skipping invalid years in file: {filename}")
            continue
        
        # Construct metdata details
        metdata_name = '_'.join([site_name, years_str, source, 'Met.nc'])
        metdata_path = os.path.join(forcing_dir, metdata_name)
        
        # Construct srfdata details
        srfdata_name = filename
        srfdata_path = os.path.join(srfdata_dir, srfdata_name)
        
        # Check if srfdata file exists
        if not os.path.exists(srfdata_path):
            print(f"Srfdata file not found: {srfdata_path}")
            continue
        
        # Read netCDF file for IGBP, lon, lat
        try:
            with Dataset(srfdata_path, 'r') as ds:
                igbp_class = str(ds['IGBP_classification'][0])
                longitude = str(ds['longitude'][0])
                latitude = str(ds['latitude'][0])
        except Exception as e:
            print(f"Error reading {srfdata_path}: {e}")
            continue
        
        # Append row
        row = [
            site_name,
            syear,
            eyear,
            source,
            srfdata_dir,
            srfdata_name,
            forcing_dir,
            metdata_name,
            igbp_class,
            longitude,
            latitude
        ]
        rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows, columns=columns)
df = df.sort_values(by='SiteName')

# Save to CSV
output_file = 'LIST_Land_selected.csv'
df.to_csv(output_file, index=False)
print(f"Statistics table saved to {output_file}")

Statistics table saved to LIST_Land_selected.csv
